# 03 · pandas Transform: groupby, join, reshape, clean

The heart of ELT in pandas: aggregate with `groupby`, combine tables with
`merge`, reshape with `pivot`/`melt`, and clean missing/dirty values. These four
skills cover most transformation work.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

In [ ]:
import pandas as pd, numpy as np
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_ts'])
customers = pd.read_csv(RAW / 'customers.csv')
items = pd.read_csv(RAW / 'order_items.csv')
products = pd.read_csv(RAW / 'products.csv')
print('loaded:', orders.shape, customers.shape, items.shape, products.shape)

## `groupby` — split, apply, combine

Group rows by a key, compute an aggregate per group, get a result table. `agg`
runs several aggregations at once and names the outputs.

In [ ]:
by_status = (orders
    .groupby('status')
    .agg(n_orders=('order_id', 'count'),
         revenue=('amount', 'sum'),
         avg_amount=('amount', 'mean'))
    .round(2)
    .reset_index())
print(by_status)

## Cleaning dirty data first

The `customers` file is deliberately messy: inconsistent country casing and some
blank emails. Clean before joining, or your groups will split (`US` vs `us`).

In [ ]:
print('before:', sorted(customers['country'].unique()))

customers['country'] = customers['country'].str.strip().str.upper()
# treat blank emails as missing and flag them
customers['email'] = customers['email'].replace('', np.nan)
customers['has_email'] = customers['email'].notna()

print('after :', sorted(customers['country'].unique()))
print('missing emails:', int((~customers['has_email']).sum()))

## `merge` — SQL-style joins

Combine tables on a key. `how=` picks the join type (`inner`, `left`, `right`,
`outer`) exactly like SQL. Here we attach each order's customer country.

In [ ]:
orders_enriched = orders.merge(
    customers[['customer_id', 'country']],
    on='customer_id', how='left')

revenue_by_country = (orders_enriched
    .query("status == 'completed'")
    .groupby('country')['amount'].sum()
    .round(2).sort_values(ascending=False))
print(revenue_by_country)

## Multi-table join + line-item revenue

Real metrics span several tables. Join line items to products to get category
revenue — the kind of star-schema roll-up you build constantly.

In [ ]:
line_rev = items.merge(products, on='product_id', how='left')
line_rev['line_amount'] = line_rev['quantity'] * line_rev['unit_price_x']

by_category = (line_rev
    .groupby('category')['line_amount'].sum()
    .round(2).sort_values(ascending=False))
print(by_category)

## Reshaping: `pivot_table` and `melt`

**Wide** vs **long** format. `pivot_table` spreads a key into columns (great for
reports); `melt` collapses columns back into rows (great for tidy storage).

In [ ]:
orders_enriched['month'] = orders_enriched['order_ts'].dt.to_period('M').astype(str)
wide = pd.pivot_table(
    orders_enriched.query("status == 'completed'"),
    index='country', columns=None, values='amount',
    aggfunc='sum', fill_value=0).round(0)
print('pivot (revenue by country):')
print(wide.head())

long = wide.reset_index().melt(id_vars='country', value_name='revenue')
print('\nmelted back to long:')
print(long.head())

## Handling missing values

`isna`, `fillna`, `dropna` are the core tools. Decide per column whether missing
means drop, fill with a default, or flag.

In [ ]:
print('nulls per column:\n', customers.isna().sum())
filled = customers.assign(email=customers['email'].fillna('UNKNOWN'))
print('\nafter fillna, nulls:', int(filled['email'].isna().sum()))

## `apply` and `map` — custom functions

When a transformation isn't a built-in vectorized op, `map` applies a function
element-wise to a **Series**, and `apply` applies one along a **DataFrame**'s
rows or columns. Prefer vectorized operations when you can (they're faster), but
these handle the arbitrary cases.

In [ ]:
# map: element-wise on a Series (here, bucket each amount)
orders['band'] = orders['amount'].map(
    lambda a: 'high' if a >= 200 else 'mid' if a >= 50 else 'low')
print(orders['band'].value_counts())

# apply along rows (axis=1): combine several columns per row
def label(row):
    return f"{row['status'][:4]}:{row['band']}"
orders['tag'] = orders.apply(label, axis=1)
print(orders[['amount', 'status', 'band', 'tag']].head())

## `concat` — stacking DataFrames

`merge` joins tables *side by side* on a key; `concat` **stacks** them — rows on
top of each other (same columns) or columns beside each other (same index).
Stacking rows is how you union daily files or append a new batch.

In [ ]:
jan = orders[orders['order_ts'].dt.month == 1].head(2)
feb = orders[orders['order_ts'].dt.month == 2].head(2)

stacked = pd.concat([jan, feb], ignore_index=True)   # union rows
print('rows:', len(jan), '+', len(feb), '->', len(stacked))
print(stacked[['order_id', 'order_ts', 'amount']])

## Duplicates: `duplicated` and `drop_duplicates`

Duplicate rows creep in from re-runs and bad joins. `duplicated` flags them;
`drop_duplicates` removes them. Use `subset=` to define "duplicate" by specific
columns and `keep=` to choose which copy to keep — essential for idempotent
loads.

In [ ]:
raw = pd.DataFrame({
    'customer_id': [1, 1, 2, 2, 2],
    'email': ['a@x.com', 'a@x.com', 'b@x.com', 'b@x.com', 'b2@x.com'],
})
print('duplicated rows:', raw.duplicated().sum())
print('after drop_duplicates:')
print(raw.drop_duplicates())
print('\none row per customer (keep last):')
print(raw.drop_duplicates(subset='customer_id', keep='last'))

### Recap

`groupby().agg()` is split-apply-combine; clean keys (casing, blanks) before
joining; `merge(how=...)` does SQL joins; chain joins for star-schema roll-ups;
`pivot_table`/`melt` switch wide↔long; `isna`/`fillna`/`dropna` handle missing
data; `map`/`apply` run custom functions; `concat` stacks frames;
`drop_duplicates` dedups for idempotent loads. Next: time-series operations.